Tasks:
1.	Preprocessing:
o	Use the training dataset as it is (do not remove stopwords).
o	Tokenize the text into words.

In [1]:
import pandas as pd

train_data = pd.read_csv('Assignment_6-train_100_wl.csv')
test_data = pd.read_csv('Assignment_6-test_20_wol.csv')

train_data['tokens'] = train_data['text'].apply(lambda x: x.split())
test_data['tokens'] = test_data['text'].apply(lambda x: x.split())

print(train_data[['text', 'tokens']].head())
print(test_data[['text', 'tokens']].head())

                           text                            tokens
0          predictable and dull          [predictable, and, dull]
1                 waste of time                 [waste, of, time]
2        story was good and fun      [story, was, good, and, fun]
3  wonderful experience overall  [wonderful, experience, overall]
4           acting was terrible           [acting, was, terrible]
                                  text  \
0                      very few laughs   
1          plain boring and uninspired   
2               story was good and fun   
3               story was good and fun   
4  excellent direction and performance   

                                     tokens  
0                       [very, few, laughs]  
1          [plain, boring, and, uninspired]  
2              [story, was, good, and, fun]  
3              [story, was, good, and, fun]  
4  [excellent, direction, and, performance]  


2.2.	Model Building:
o	Calculate prior probabilities of each class (+ve and -ve).
o	Calculate likelihood probabilities of each word given a class using Naïve Bayes formula.


In [2]:
positive_docs = train_data[train_data['category'] == 'positive']
negative_docs = train_data[train_data['category'] == 'negative']
total_docs = len(train_data)
p_positive = len(positive_docs) / total_docs
p_negative = len(negative_docs) / total_docs
positive_words = []
for tokens in positive_docs['tokens']:
    positive_words.extend(tokens)

negative_words = []
for tokens in negative_docs['tokens']:
    negative_words.extend(tokens)

vocab = set(positive_words + negative_words)
V = len(vocab)
positive_word_counts = {}
for word in positive_words:
    positive_word_counts[word] = positive_word_counts.get(word, 0) + 1
negative_word_counts = {}
for word in negative_words:
    negative_word_counts[word] = negative_word_counts.get(word, 0) + 1

total_positive_words = len(positive_words)
total_negative_words = len(negative_words)
word_likelihoods = {}
for word in vocab:
    p_word_positive = (positive_word_counts.get(word, 0) + 1) / (total_positive_words + V)
    p_word_negative = (negative_word_counts.get(word, 0) + 1) / (total_negative_words + V)
    word_likelihoods[word] = (p_word_positive, p_word_negative)

print(f"Prior probabilities:")
print(f"P(positive) = {p_positive:.4f}")
print(f"P(negative) = {p_negative:.4f}")
print(f"\nVocabulary size: {V}")
print(f"\nSample word likelihoods:")
for word in list(vocab)[:5]:
    print(f"'{word}': P(+ve)={word_likelihoods[word][0]:.6f}, P(-ve)={word_likelihoods[word][1]:.6f}")

Prior probabilities:
P(positive) = 0.5000
P(negative) = 0.5000

Vocabulary size: 51

Sample word likelihoods:
'powerful': P(+ve)=0.029740, P(-ve)=0.004484
'characters': P(+ve)=0.029740, P(-ve)=0.004484
'disappointing': P(+ve)=0.003717, P(-ve)=0.031390
'excellent': P(+ve)=0.014870, P(-ve)=0.004484
'very': P(+ve)=0.029740, P(-ve)=0.017937


3. 3.	Prediction:
o	Apply your Naïve Bayes model on the test dataset.
o	Assign each test sentence either +ve or -ve sentiment

In [4]:
import numpy as np

def predict_sentiment(tokens):
    log_prob_positive = np.log(p_positive)
    log_prob_negative = np.log(p_negative)

    for word in tokens:
        if word in word_likelihoods:
            p_word_pos, p_word_neg = word_likelihoods[word]
            log_prob_positive += np.log(p_word_pos)
            log_prob_negative += np.log(p_word_neg)

    return 'positive' if log_prob_positive > log_prob_negative else 'negative'
test_data['predicted'] = test_data['tokens'].apply(predict_sentiment)
print(test_data[['text', 'predicted']])

                                   text predicted
0                       very few laughs  negative
1           plain boring and uninspired  negative
2                story was good and fun  positive
3                story was good and fun  positive
4   excellent direction and performance  positive
5        amazing film with great acting  positive
6                      not a good story  positive
7           plain boring and uninspired  negative
8           plain boring and uninspired  negative
9   the film was touching and inspiring  positive
10       good storyline and nice acting  positive
11            movie was boring and slow  negative
12             really enjoyed the movie  positive
13                        waste of time  negative
14                     not a good story  positive
15  the film was touching and inspiring  positive
16       poor direction and weak script  negative
17       loved the characters and story  positive
18  excellent direction and performance  positive


4. Evaluation:
o	Calculate Precision, Recall, Accuracy and F1-score without any package.
o	Calculate Precision, Recall, Accuracy and F1-score with package.
o	Is there any difference between these two ways of evaluation? If yes, please mention that.

In [5]:
test_data_with_labels = pd.read_csv('Assignment_6-test_20_wl.csv')
actual = test_data_with_labels['category']
predicted = test_data['predicted']

TP = sum((actual == 'positive') & (predicted == 'positive'))
TN = sum((actual == 'negative') & (predicted == 'negative'))
FP = sum((actual == 'negative') & (predicted == 'positive'))
FN = sum((actual == 'positive') & (predicted == 'negative'))

precision_manual = TP / (TP + FP) if (TP + FP) > 0 else 0
recall_manual = TP / (TP + FN) if (TP + FN) > 0 else 0
accuracy_manual = (TP + TN) / len(actual)
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual) if (precision_manual + recall_manual) > 0 else 0

print("Manual Calculation:")
print(f"Precision: {precision_manual:.4f}")
print(f"Recall: {recall_manual:.4f}")
print(f"Accuracy: {accuracy_manual:.4f}")
print(f"F1-Score: {f1_manual:.4f}")

from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

precision_package = precision_score(actual, predicted, pos_label='positive')
recall_package = recall_score(actual, predicted, pos_label='positive')
accuracy_package = accuracy_score(actual, predicted)
f1_package = f1_score(actual, predicted, pos_label='positive')

print("\nPackage Calculation:")
print(f"Precision: {precision_package:.4f}")
print(f"Recall: {recall_package:.4f}")
print(f"Accuracy: {accuracy_package:.4f}")
print(f"F1-Score: {f1_package:.4f}")

print("\nDifference Analysis:")
print(f"Precision difference: {abs(precision_manual - precision_package):.6f}")
print(f"Recall difference: {abs(recall_manual - recall_package):.6f}")
print(f"Accuracy difference: {abs(accuracy_manual - accuracy_package):.6f}")
print(f"F1-Score difference: {abs(f1_manual - f1_package):.6f}")

if any([abs(precision_manual - precision_package) > 1e-10,
        abs(recall_manual - recall_package) > 1e-10,
        abs(accuracy_manual - accuracy_package) > 1e-10,
        abs(f1_manual - f1_package) > 1e-10]):
    print("\nThere is a difference between manual and package calculations.")
else:
    print("\nNo difference between manual and package calculations.")

Manual Calculation:
Precision: 0.8333
Recall: 1.0000
Accuracy: 0.9000
F1-Score: 0.9091

Package Calculation:
Precision: 0.8333
Recall: 1.0000
Accuracy: 0.9000
F1-Score: 0.9091

Difference Analysis:
Precision difference: 0.000000
Recall difference: 0.000000
Accuracy difference: 0.000000
F1-Score difference: 0.000000

No difference between manual and package calculations.
